# Error Type Analysis — Partial Reversal vs Others
### Digit Span Backwards | DBS ATN Case 1 | Sessions 2 & 3

---

## ⚠️ Condition B is active throughout this notebook

**Classification logic (Condition B, mA ≥ 2.0):**
- `stim_present_condA` = any packet ≥ 2.0 mA anywhere in trial
- `feedback_only_stim` = stim present but ONLY in feedback window (`pre_fb_frac < 0.3` AND `fb_frac ≥ 0.3`)
- `stim_present_condB` = `stim_present_condA AND NOT feedback_only_stim`  ← **used everywhere**

---

## Error Taxonomy (wrong trials only)

| Category | Definition | Example |
|---|---|---|
| **PARTIAL_REVERSAL** | Patient starts reversing correctly (`resp[0] == presented[-1]`) but full response is wrong | 568 → 856 (partially reversed) |
| **OTHER** | No clear reversal pattern; all digits in response are from the presented set (permutation error) | 5463 → 4653 |
| ~~DIFFERENT_NUMBER~~ | Response contains a digit not in the presented sequence | 352 → 256 — **excluded from analysis** |

---

## Plot Structure

1. **Plot 1** — Total wrong trials per session (S2, S3, Combined) — pure count bar, no stats
2. **Plot 2** — Wrong trial breakdown: PARTIAL_REVERSAL vs OTHER — count bars, no stats
3. **Plot 3** — PARTIAL_REVERSAL: Stim-ON vs Stim-OFF (S2 | S3 | Combined) — with stats
4. **Plot 4** — OTHER errors: Stim-ON vs Stim-OFF (S2 | S3 | Combined) — with stats
5. **Plot 5** — PARTIAL_REVERSAL rate vs OTHER rate by stim group — stacked/grouped bar

In [ ]:
# ── Cell 1: Imports & Display Settings ───────────────────────────────────────
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from scipy import stats
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams.update({
    'font.family':       'sans-serif',
    'font.size':         10,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'figure.dpi':        130,
    'savefig.dpi':       160,
    'savefig.bbox':      'tight',
    'savefig.facecolor': 'white',
})

# ── Colour palette ────────────────────────────────────────────────────────────
C_NO_STIM   = '#90A4AE'   # grey  — no stimulation
C_STIM      = '#1A56DB'   # blue  — stimulation present
C_CORRECT   = '#2E7D32'   # green
C_WRONG     = '#C62828'   # red
C_S2        = '#1565C0'   # Session 2 (dark blue)
C_S3        = '#00838F'   # Session 3 (teal)
C_PARTIAL   = '#E65100'   # orange — partial reversal
C_OTHER     = '#6A1B9A'   # purple — other errors
C_EXCL      = '#B0BEC5'   # light grey — different-number (excluded)

# ── Constants ─────────────────────────────────────────────────────────────────
STIM_THRESHOLD = 2.0   # mA — Condition B threshold
N_TRIALS_PER_SESSION = 14

print('Imports OK  |  Condition B active  |  STIM_THRESHOLD =', STIM_THRESHOLD, 'mA')

In [ ]:
# ── Cell 2: File Paths  ← EDIT THESE ─────────────────────────────────────────

JSON_PATH_S2   = Path(r"C:\Users\ASSUS\ATN\Digit Span Backwards\Data\Neural Data\DBS ATN DSB Case 1\D. Siragusa\3.5.26\Time stamp 1433\Report_Json_Session_Report_20260305T151703.json")
CSV_PATH_S2    = Path(r"C:\Users\ASSUS\ATN\Digit Span Backwards\Data\Eprime Data\Digit Span Backwards v3.2\DigitSpanBackward v3.3-6-2-Scores.csv")
EVENTS_PATH_S2 = Path(r"C:\Users\ASSUS\ATN\Digit Span Backwards\Session 2\Events.csv")
OUT_DIR_S2     = Path(r"C:\Users\ASSUS\ATN\Digit Span Backwards\Session 2")

JSON_PATH_S3   = Path(r"C:\Users\ASSUS\ATN\Digit Span Backwards\Data\Neural Data\DBS ATN DSB Case 1\D. Siragusa\3.5.26\Time stamp 1441\Report_Json_Session_Report_20260305T151912.json")
CSV_PATH_S3    = Path(r"C:\Users\ASSUS\ATN\Digit Span Backwards\Data\Eprime Data\Digit Span Backwards v3.2\DigitSpanBackward v3.3-6-3-Scores.csv")
EVENTS_PATH_S3 = Path(r"C:\Users\ASSUS\ATN\Digit Span Backwards\Session 3\Preprocessed Data\Events.csv")
OUT_DIR_S3     = Path(r"C:\Users\ASSUS\ATN\Digit Span Backwards\Session 3")

COMBINED_DIR   = Path(r"C:\Users\ASSUS\ATN\Digit Span Backwards\Session 2 v 3\Base\Error Type Analysis\Visuals")

COMBINED_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR_S2.mkdir(parents=True, exist_ok=True)
OUT_DIR_S3.mkdir(parents=True, exist_ok=True)

print('Paths set.')

In [ ]:
# ── Cell 3: Data Loading & Trial Classification Functions ─────────────────────

def load_session_data(json_path, csv_path, events_path):
    """Load JSON LFP report, E-Prime CSV, and Events CSV for one session."""
    with open(json_path) as f:
        report = json.load(f)
    eprime_df = pd.read_csv(csv_path,    encoding='utf-8-sig', low_memory=False)
    ev_df     = pd.read_csv(events_path, encoding='utf-8-sig', low_memory=False)

    meta = dict(
        subject = str(eprime_df['Subject'].iloc[0]),
        session = str(eprime_df['Session'].iloc[0]),
        date    = str(eprime_df['SessionDate'].iloc[0]),
    )

    # ── Sync anchor: first 0 → +mA transition in BrainSenseLfp ──────────────
    stim_tick = None
    for stream in report['BrainSenseLfp']:
        prev = None
        for pkt in stream['LfpData']:
            curr = pkt['Left']['mA']
            if prev is not None and prev == 0.0 and curr > 0.0:
                stim_tick = pkt['TicksInMs']
                break
            prev = curr
        if stim_tick:
            break
    assert stim_tick, 'No 0->+mA transition found in BrainSenseLfp!'

    welcome_ms    = int(eprime_df['Welcome.TargetOnsetTime'].iloc[0])
    MANUAL_OFFSET = stim_tick - welcome_ms
    def to_rel(ms): return float(ms) + MANUAL_OFFSET - stim_tick

    # ── Build LFP timeline (sorted) ───────────────────────────────────────────
    ticks, mAs = [], []
    for stream in report['BrainSenseLfp']:
        for pkt in stream['LfpData']:
            ticks.append(pkt['TicksInMs'])
            mAs.append(pkt['Left']['mA'])
    ticks = np.array(ticks, dtype=float)
    mAs   = np.array(mAs,   dtype=float)
    order = np.argsort(ticks)
    ticks_rel = ticks[order] - stim_tick
    mAs       = mAs[order]

    print(f"Loaded: Subject={meta['subject']}  Session={meta['session']}  "
          f"Date={meta['date']}  mA_max={mAs.max():.2f}  N_packets={len(ticks)}")
    return report, eprime_df, ev_df, meta, to_rel, ticks_rel, mAs


def stim_frac_in_window(t0, t1, ticks_rel, mAs, threshold=2.0):
    """Fraction of window duration (trapezoidal) where mA >= threshold."""
    if t0 is None or t1 is None or t1 <= t0:
        return 0.0
    mask = (ticks_rel >= t0) & (ticks_rel <= t1)
    t_r, t_m = ticks_rel[mask], mAs[mask]
    if len(t_r) < 2:
        return 0.0
    dt    = np.diff(t_r)
    mid   = (t_m[:-1] + t_m[1:]) / 2.0
    total = t_r[-1] - t_r[0]
    return float(np.sum(dt[mid >= threshold]) / total) if total > 0 else 0.0


def peak_mA(t0, t1, ticks_rel, mAs):
    """Peak mA in window; falls back to nearest packet if window has no samples."""
    if t0 is None or t1 is None or t1 <= t0:
        return 0.0
    mask = (ticks_rel >= t0) & (ticks_rel <= t1)
    if not mask.any():
        # Nearest-packet fallback (LFP step-hold encoding)
        idx = int(np.argmin(np.abs(ticks_rel - (t0 + t1) / 2)))
        return float(mAs[idx])
    return float(mAs[mask].max())


def build_all_trials(eprime_df, ev_df, to_rel, ticks_rel, mAs, sess_label):
    """Build one trial dict per trial, including Condition A and Condition B classification."""
    digit_rows = eprime_df['Digit'].tolist()
    trial_digit_seqs = {}
    offset = 0
    for tn in range(1, 15):
        row = ev_df[(ev_df['Event_Type'] == 'Main Trial Start') & (ev_df['Trial_Number'] == tn)]
        if row.empty:
            continue
        n = int(row.iloc[0]['Num_Digits'])
        trial_digit_seqs[tn] = digit_rows[offset:offset + n]
        offset += n

    def ev_all(etype, tn):
        return ev_df[(ev_df['Event_Type'] == etype) &
                     (ev_df['Trial_Number'] == tn)]['Time_ms'].tolist()
    def ev_first(etype, tn):
        v = ev_all(etype, tn)
        return v[0] if v else None

    trials = []
    for tn in range(1, 15):
        sr = ev_df[(ev_df['Event_Type'] == 'Main Trial Start') & (ev_df['Trial_Number'] == tn)]
        if sr.empty:
            continue
        r = sr.iloc[0]

        acc     = int(r['ACC'])        if pd.notna(r['ACC'])        else None
        cresp_v = r['CRESP']           if pd.notna(r['CRESP'])      else None
        resp_v  = r['RESP']            if pd.notna(r['RESP'])       else None
        nd      = int(r['Num_Digits']) if pd.notna(r['Num_Digits']) else None
        if None in (acc, cresp_v, resp_v, nd):
            continue

        cresp_s   = str(int(cresp_v)).zfill(nd)
        resp_s    = str(int(resp_v)).zfill(nd)
        presented = cresp_s[::-1]   # what was shown on screen

        # ── Event timestamps → neural-relative ms ─────────────────────────────
        t_start_ms  = ev_first('Main Trial Start', tn)
        t_end_ms    = ev_first('Main Trial End',   tn)
        t_start     = to_rel(t_start_ms) if t_start_ms else None
        t_end       = to_rel(t_end_ms)   if t_end_ms   else None

        fix_starts  = [to_rel(ms) for ms in ev_all('Fixation Start', tn)]
        stim_starts = [to_rel(ms) for ms in ev_all('Stimulus Start', tn)]
        stim_ends   = [to_rel(ms) for ms in ev_all('Stimulus End',   tn)]
        cs_ms = ev_first('Choice Start',   tn); cs = to_rel(cs_ms) if cs_ms else None
        ce_ms = ev_first('Choice End',     tn); ce = to_rel(ce_ms) if ce_ms else None
        fs_ms = ev_first('Feedback Start', tn); fs = to_rel(fs_ms) if fs_ms else None
        fe_ms = ev_first('Feedback End',   tn); fe = to_rel(fe_ms) if fe_ms else None

        # ── Condition A: any stim ≥ threshold anywhere in trial ───────────────
        trial_lo = fix_starts[0] if fix_starts else t_start
        trial_hi = fe if fe else t_end
        mask_full = (ticks_rel >= trial_lo) & (ticks_rel <= trial_hi)
        stim_present_condA = bool(mask_full.any() and np.any(mAs[mask_full] >= STIM_THRESHOLD))

        # ── Condition B: exclude feedback-only stim ───────────────────────────
        pre_fb_frac = 0.0
        if fix_starts and stim_starts:
            pre_fb_frac = max(pre_fb_frac,
                stim_frac_in_window(fix_starts[0], stim_starts[0], ticks_rel, mAs, STIM_THRESHOLD))
        if stim_starts and stim_ends:
            pre_fb_frac = max(pre_fb_frac,
                stim_frac_in_window(stim_starts[0], stim_ends[0], ticks_rel, mAs, STIM_THRESHOLD))
        if cs and ce:
            pre_fb_frac = max(pre_fb_frac,
                stim_frac_in_window(cs, ce, ticks_rel, mAs, STIM_THRESHOLD))
        fb_frac = stim_frac_in_window(fs, fe, ticks_rel, mAs, STIM_THRESHOLD) if fs and fe else 0.0

        feedback_only_stim = stim_present_condA and (pre_fb_frac < 0.3) and (fb_frac >= 0.3)
        stim_present_condB = stim_present_condA and not feedback_only_stim

        # ── Per-window peak mA ────────────────────────────────────────────────
        fix_peak  = max((peak_mA(fs2, ss, ticks_rel, mAs)
                         for fs2, ss in zip(fix_starts, stim_starts)), default=0.0)
        stim_peak = max((peak_mA(ss, se, ticks_rel, mAs)
                         for ss, se in zip(stim_starts, stim_ends)), default=0.0)
        ch_peak   = peak_mA(cs, ce, ticks_rel, mAs)
        fb_peak   = peak_mA(fs, fe, ticks_rel, mAs)
        whole_peak = peak_mA(trial_lo, trial_hi, ticks_rel, mAs)

        trials.append(dict(
            session              = sess_label,
            trial_num            = tn,
            digits               = nd,
            acc                  = acc,
            presented            = presented,
            cresp                = cresp_s,
            resp                 = resp_s,
            digit_seq            = trial_digit_seqs.get(tn, []),
            stim_present         = stim_present_condA,    # Condition A
            stim_present_condB   = stim_present_condB,    # Condition B ← primary
            feedback_only_stim   = feedback_only_stim,
            whole_peak_mA        = round(whole_peak, 3),
            fix_peak_mA          = round(fix_peak,   3),
            stim_peak_mA         = round(stim_peak,  3),
            ch_peak_mA           = round(ch_peak,    3),
            fb_peak_mA           = round(fb_peak,    3),
        ))

    return trials


print('Core functions defined.')

In [ ]:
# ── Cell 4: Statistical Helper Functions ─────────────────────────────────────

def cohens_d(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    if len(a) < 2 or len(b) < 2:
        return np.nan
    pooled = np.sqrt(((len(a)-1)*np.var(a, ddof=1) +
                      (len(b)-1)*np.var(b, ddof=1)) / (len(a)+len(b)-2))
    return float((np.mean(a) - np.mean(b)) / pooled) if pooled > 0 else np.nan


def effect_label(d):
    if np.isnan(d): return 'N/A'
    v = abs(d)
    if v >= 0.8: return 'large'
    if v >= 0.5: return 'medium'
    if v >= 0.2: return 'small'
    return 'negligible'


def sig_stars(p):
    if np.isnan(p): return ''
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'ns'


def run_fisher_comparison(group_no, group_on, label):
    """
    Compare rate of a binary outcome (e.g. is_partial) between
    stim-OFF and stim-ON groups using Fisher's exact test + Cohen's d.
    Each group is a list of 0/1 values.
    """
    a = np.asarray(group_no, float)
    b = np.asarray(group_on, float)
    n_no, n_on = len(a), len(b)
    c_no, c_on = int(a.sum()), int(b.sum())

    pct_no = a.mean() * 100 if n_no > 0 else np.nan
    pct_on = b.mean() * 100 if n_on > 0 else np.nan

    if n_no > 0 and n_on > 0:
        _, p = stats.fisher_exact([
            [c_no,   n_no - c_no],
            [c_on,   n_on - c_on]
        ])
        p = float(p)
    else:
        p = np.nan

    d = cohens_d(a, b)

    return dict(
        label       = label,
        n_no_stim   = n_no,
        count_no    = c_no,
        pct_no      = pct_no,
        n_stim      = n_on,
        count_on    = c_on,
        pct_on      = pct_on,
        p_value     = p,
        stars       = sig_stars(p),
        cohens_d    = d,
        effect_size = effect_label(d),
    )


def _stat_box(ax, res, y_pos, fontsize=9):
    """Draw stats annotation box on an axes panel."""
    p_v = res['p_value']
    d_v = res['cohens_d']
    eff = effect_label(d_v)
    sig = sig_stars(p_v)
    p_str = f'p={p_v:.3f}' if not np.isnan(p_v) else 'p=N/A'
    d_str = f'd={d_v:+.2f}' if not np.isnan(d_v) else 'd=N/A'
    notable = ((not np.isnan(p_v) and p_v < 0.05) or
               (not np.isnan(d_v) and abs(d_v) >= 0.8))
    fc = '#FFF0F0' if notable else '#F5F5F5'
    ec = '#C62828' if notable else '#888888'
    txt = f'{sig}  {p_str}  {d_str}  [{eff}]'
    ax.text(0.5, y_pos, txt,
            transform=ax.transAxes, ha='center', va='center',
            fontsize=fontsize, fontweight='bold', color=ec,
            bbox=dict(boxstyle='round,pad=0.35', fc=fc, ec=ec, lw=1.2, alpha=0.97))


def save_fig(fig, path):
    fig.savefig(path, bbox_inches='tight', dpi=160, facecolor='white')
    plt.show()
    plt.close(fig)
    print(f'Saved -> {path}')


print('Stats helpers defined.')

In [ ]:
# ── Cell 5: Load Sessions & Classify Trials ───────────────────────────────────

# Session 2
_, ep_s2, ev_s2, meta_s2, to_rel_s2, tr_s2, mA_s2 = \
    load_session_data(JSON_PATH_S2, CSV_PATH_S2, EVENTS_PATH_S2)
all_s2 = build_all_trials(ep_s2, ev_s2, to_rel_s2, tr_s2, mA_s2, 'Session 2')

print()

# Session 3
_, ep_s3, ev_s3, meta_s3, to_rel_s3, tr_s3, mA_s3 = \
    load_session_data(JSON_PATH_S3, CSV_PATH_S3, EVENTS_PATH_S3)
all_s3 = build_all_trials(ep_s3, ev_s3, to_rel_s3, tr_s3, mA_s3, 'Session 3')

# Combined
all_combined = all_s2 + all_s3

print(f'\nTotal trials — S2: {len(all_s2)}, S3: {len(all_s3)}, Combined: {len(all_combined)}')

In [ ]:
# ── Cell 6: Error Type Classification Function & Trial Inventory ──────────────
#
# Three mutually exclusive categories (wrong trials only):
#
#   PARTIAL_REVERSAL  — resp[0] == presented[-1]  (started reversing correctly
#                        but did not complete the full reversal)
#                        Example: presented=568, resp=856 (resp starts with 8 = last digit of 568)
#
#   OTHER             — no clear reversal pattern; all digits in resp are a
#                        subset / permutation of digits in presented set
#                        (general scrambling without reversal attempt)
#                        Example: presented=5463, resp=4653
#
#   DIFFERENT_NUMBER  — resp contains at least one digit not in the presented set
#                        Example: presented=352, resp=256 (digit 6 was not in 352)
#                        *** EXCLUDED FROM ALL ANALYSES ***

def classify_error(presented, resp):
    """
    Classify a single wrong trial.

    Parameters
    ----------
    presented : str   e.g. '568'
    resp      : str   e.g. '856'

    Returns
    -------
    'PARTIAL_REVERSAL' | 'OTHER' | 'DIFFERENT_NUMBER'
    """
    presented = str(presented).strip()
    resp      = str(resp).strip()

    # Zero-pad to same length if needed
    n = max(len(presented), len(resp))
    presented = presented.zfill(n)
    resp      = resp.zfill(n)

    presented_digits = sorted(presented)   # multiset for membership check
    resp_digits      = sorted(resp)

    # DIFFERENT_NUMBER: resp uses a digit not available in presented
    if resp_digits != presented_digits:
        return 'DIFFERENT_NUMBER'

    # PARTIAL_REVERSAL: patient started from the correct end
    # i.e. resp[0] == presented[-1]  (first response digit = last presented digit)
    if resp[0] == presented[-1]:
        return 'PARTIAL_REVERSAL'

    # OTHER: permutation error with no reversal attempt
    return 'OTHER'


def annotate_error_types(trials):
    """Add 'error_type' field to each trial dict (in-place). Correct trials get 'CORRECT'."""
    for t in trials:
        if t['acc'] == 1:
            t['error_type'] = 'CORRECT'
        else:
            t['error_type'] = classify_error(t['presented'], t['resp'])
    return trials


# Annotate all sessions
annotate_error_types(all_s2)
annotate_error_types(all_s3)
# all_combined shares the same dicts so already updated


# ── Inventory printout ────────────────────────────────────────────────────────
def print_error_inventory(trials, sess_label):
    print('='*80)
    print(f'  {sess_label}  —  Error Type Inventory  |  Condition B  |  mA >= {STIM_THRESHOLD}')
    print('='*80)
    hdr = (f'  {"Trial":>5}  {"Digits":>6}  {"StimB":>5}  {"PeakMa":>7}  '
           f'{"Presented":>10}  {"Response":>10}  {"ErrorType":>18}')
    print(hdr)
    print('  ' + '-'*75)
    counts = {'CORRECT': 0, 'PARTIAL_REVERSAL': 0, 'OTHER': 0, 'DIFFERENT_NUMBER': 0}
    for t in trials:
        stim_str = 'ON ' if t['stim_present_condB'] else 'OFF'
        et = t['error_type']
        counts[et] += 1
        marker = '  [EXCL]' if et == 'DIFFERENT_NUMBER' else ''
        print(f'  T{t["trial_num"]:>4}  {t["digits"]:>6}  {stim_str:>5}  '
              f'{t["whole_peak_mA"]:>7.3f}  '
              f'{t["presented"]:>10}  {t["resp"]:>10}  {et:>18}{marker}')
    print()
    print(f'  Summary: CORRECT={counts["CORRECT"]}  '
          f'PARTIAL_REVERSAL={counts["PARTIAL_REVERSAL"]}  '
          f'OTHER={counts["OTHER"]}  '
          f'DIFFERENT_NUMBER={counts["DIFFERENT_NUMBER"]} (excluded)')
    wrong_incl = counts['PARTIAL_REVERSAL'] + counts['OTHER']
    print(f'  Total wrong (included in analysis): {wrong_incl} / {len(trials)}')
    print()


print_error_inventory(all_s2, 'Session 2')
print_error_inventory(all_s3, 'Session 3')

In [ ]:
# ── Cell 7: Statistical Comparison Printout ───────────────────────────────────
# Compare PARTIAL_REVERSAL rate and OTHER rate between Stim-OFF vs Stim-ON
# using Condition B classification.

def get_wrong_included(trials):
    """Return wrong trials that are included in analysis (exclude DIFFERENT_NUMBER)."""
    return [t for t in trials if t['error_type'] in ('PARTIAL_REVERSAL', 'OTHER')]


def stats_summary(trials, sess_label):
    wrong = get_wrong_included(trials)
    no_stim = [t for t in wrong if not t['stim_present_condB']]
    stim_on = [t for t in wrong if     t['stim_present_condB']]

    print('='*65)
    print(f'  {sess_label}  |  Condition B  |  mA >= {STIM_THRESHOLD}')
    print(f'  Wrong trials (excl. DIFFERENT_NUMBER): {len(wrong)}')
    print(f'  NO-STIM: {len(no_stim)} trials  |  STIM-ON: {len(stim_on)} trials')
    print('='*65)

    for err_type, label in [('PARTIAL_REVERSAL', 'Partial Reversal'), ('OTHER', 'Other')]:
        vec_no = [1 if t['error_type'] == err_type else 0 for t in no_stim]
        vec_on = [1 if t['error_type'] == err_type else 0 for t in stim_on]
        res = run_fisher_comparison(vec_no, vec_on, label)

        p_s = f'{res["p_value"]:.4f}' if not np.isnan(res['p_value']) else 'N/A'
        d_s = f'{res["cohens_d"]:+.3f}' if not np.isnan(res['cohens_d']) else 'N/A'
        print(f'\n  [{label}]')
        print(f'    NO-STIM: {res["count_no"]}/{res["n_no_stim"]} ({res["pct_no"]:.1f}%)')
        print(f'    STIM-ON: {res["count_on"]}/{res["n_stim"]} ({res["pct_on"]:.1f}%)')
        print(f'    Fisher p: {p_s}  {res["stars"]}   Cohen\'s d: {d_s}   [{res["effect_size"]}]')
    print()


stats_summary(all_s2,       'Session 2')
stats_summary(all_s3,       'Session 3')
stats_summary(all_combined, 'Combined (S2 + S3)')

In [ ]:
# ── Cell 8: Plot 1 — Total Wrong Trials per Session (pure count, no stats) ────
# Bars: S2 | S3 | Combined
# Shows: correct, PARTIAL_REVERSAL, OTHER, DIFFERENT_NUMBER (excluded, shown greyed)
# y-axis: out of 14 (or 28 for combined)

def count_types(trials):
    c = {'CORRECT': 0, 'PARTIAL_REVERSAL': 0, 'OTHER': 0, 'DIFFERENT_NUMBER': 0}
    for t in trials:
        c[t['error_type']] += 1
    return c


sessions_info = [
    ('Session 2',         all_s2,       14),
    ('Session 3',         all_s3,       14),
    ('Combined (S2+S3)',  all_combined, 28),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5.5), facecolor='white')
fig.subplots_adjust(wspace=0.38, top=0.82, bottom=0.13, left=0.07, right=0.97)

bar_cats = [
    ('CORRECT',          C_CORRECT,  'Correct'),
    ('PARTIAL_REVERSAL', C_PARTIAL,  'Partial Reversal'),
    ('OTHER',            C_OTHER,    'Other Error'),
    ('DIFFERENT_NUMBER', C_EXCL,     'Different Number (excl.)'),
]

for ax, (sess_label, trials, n_total) in zip(axes, sessions_info):
    c = count_types(trials)
    wrong_total = c['PARTIAL_REVERSAL'] + c['OTHER']   # included wrong

    xs     = np.arange(len(bar_cats))
    colors = [bc[1] for bc in bar_cats]
    vals   = [c[bc[0]] for bc in bar_cats]

    bars = ax.bar(xs, vals, width=0.55, color=colors,
                  edgecolor='white', linewidth=1.4, zorder=3)

    # Labels inside/above bars
    for xi, v in enumerate(vals):
        pct = 100 * v / n_total
        label_txt = f'{v}/{n_total}\n({pct:.0f}%)'
        if v >= 2:
            ax.text(xi, v / 2, label_txt, ha='center', va='center',
                    fontsize=9, fontweight='bold', color='white', zorder=8)
        elif v > 0:
            ax.text(xi, v + 0.3, label_txt, ha='center', va='bottom',
                    fontsize=8.5, fontweight='bold', color=colors[xi], zorder=8)
        else:
            ax.text(xi, 0.3, '0', ha='center', va='bottom',
                    fontsize=9, fontweight='bold', color='#aaa', zorder=8)

    ax.set_xticks(xs)
    ax.set_xticklabels([bc[2] for bc in bar_cats], fontsize=8.5, fontweight='bold',
                       rotation=15, ha='right')
    ax.set_ylim(0, n_total + 3)
    ax.set_yticks(range(0, n_total + 1, 2))
    ax.set_ylabel('Number of Trials', fontsize=9)
    ax.yaxis.grid(True, color='#eeeeee', zorder=0)
    ax.set_axisbelow(True)
    ax.set_facecolor('#FAFAFA')
    ax.set_title(sess_label, fontsize=11, fontweight='bold', color='#1A1A2E', pad=8)

    # Annotate total wrong (included)
    ax.text(0.98, 0.97, f'Wrong (incl.): {wrong_total}/{n_total}',
            transform=ax.transAxes, ha='right', va='top',
            fontsize=8, color=C_WRONG,
            bbox=dict(boxstyle='round,pad=0.3', fc='#FFF0F0', ec=C_WRONG, lw=0.8))

fig.suptitle('Plot 1 — Trial Outcome Breakdown per Session',
             fontsize=13, fontweight='bold', color='#111', y=0.97)
fig.text(0.5, 0.88,
         'Condition B  |  mA ≥ 2.0  |  Pure counts out of 14 per session  |  '
         'DIFFERENT_NUMBER trials excluded from error analysis',
         ha='center', fontsize=8.5, color='#555')

legend_handles = [
    mpatches.Patch(facecolor=C_CORRECT, label='Correct'),
    mpatches.Patch(facecolor=C_PARTIAL, label='Partial Reversal'),
    mpatches.Patch(facecolor=C_OTHER,   label='Other Error'),
    mpatches.Patch(facecolor=C_EXCL,    label='Different Number (excluded)'),
]
fig.legend(handles=legend_handles, loc='lower center', ncol=4,
           fontsize=9, framealpha=0.97, facecolor='white',
           edgecolor='#ccc', bbox_to_anchor=(0.5, 0.01))

save_fig(fig, COMBINED_DIR / 'plot1_trial_outcome_breakdown_condB.png')

In [ ]:
# ── Cell 9: Plot 2 — Wrong Trial Breakdown: PARTIAL_REVERSAL vs OTHER ─────────
# Scope: wrong trials only (DIFFERENT_NUMBER excluded)
# Layout: 1×3  (S2 | S3 | Combined)
# No stats — pure count

fig, axes = plt.subplots(1, 3, figsize=(13, 5), facecolor='white')
fig.subplots_adjust(wspace=0.38, top=0.82, bottom=0.13, left=0.07, right=0.97)

for ax, (sess_label, trials, n_total) in zip(axes, sessions_info):
    wrong = [t for t in trials if t['error_type'] in ('PARTIAL_REVERSAL', 'OTHER')]
    n_partial = sum(1 for t in wrong if t['error_type'] == 'PARTIAL_REVERSAL')
    n_other   = sum(1 for t in wrong if t['error_type'] == 'OTHER')
    n_wrong   = len(wrong)

    vals   = [n_partial, n_other]
    colors = [C_PARTIAL, C_OTHER]
    labels = ['Partial\nReversal', 'Other\nError']

    bars = ax.bar([0, 1], vals, width=0.5, color=colors,
                  edgecolor='white', linewidth=1.4, zorder=3)

    for xi, (v, col) in enumerate(zip(vals, colors)):
        pct_of_wrong  = (100 * v / n_wrong)  if n_wrong  > 0 else 0
        pct_of_total  = (100 * v / n_total)
        lbl = f'{v}/{n_wrong}'

        if v >= 2:
            ax.text(xi, v / 2, lbl, ha='center', va='center',
                    fontsize=9, fontweight='bold', color='black', zorder=8)
        elif v > 0:
            ax.text(xi, v + 0.2, lbl, ha='center', va='bottom',
                    fontsize=8.5, fontweight='bold', color=col)
        else:
            ax.text(xi, 0.2, '0', ha='center', va='bottom',
                    fontsize=9, color='#aaa')

    ax.set_xticks([0, 1])
    ax.set_xticklabels(labels, fontsize=10, fontweight='bold')
    ax.set_ylim(0, n_total + 3)
    ax.set_yticks(range(0, n_total + 1, 2))
    ax.set_ylabel('Number of Wrong Trials', fontsize=9)
    ax.yaxis.grid(True, color='#eeeeee', zorder=0)
    ax.set_axisbelow(True)
    ax.set_facecolor('#FAFAFA')
    ax.set_title(sess_label, fontsize=11, fontweight='bold', color='#1A1A2E', pad=8)

    ax.text(0.98, 0.97, f'Total wrong: {n_wrong}/{n_total}',
            transform=ax.transAxes, ha='right', va='top', fontsize=8, color=C_WRONG,
            bbox=dict(boxstyle='round,pad=0.3', fc='#FFF0F0', ec=C_WRONG, lw=0.8))

fig.suptitle('Plot 2 — Wrong Trial Breakdown: Partial Reversal vs Other',
             fontsize=13, fontweight='bold', color='#111', y=0.97)

fig.text(0.5, 0.88,
         'Condition B  |  mA ≥ 2.0  |  Wrong trials only (DIFFERENT_NUMBER excluded)  |  No statistics',
         ha='center', fontsize=8.5, color='#555')

fig.legend(handles=[
    mpatches.Patch(facecolor=C_PARTIAL, label='Partial Reversal'),
    mpatches.Patch(facecolor=C_OTHER,   label='Other Error'),
], loc='lower center', ncol=2, fontsize=10, framealpha=0.97,
   facecolor='white', edgecolor='#ccc', bbox_to_anchor=(0.5, 0.01))

save_fig(fig, COMBINED_DIR / 'plot2_wrong_breakdown_condB.png')

In [ ]:
# ── Cell 10: Plot 3 — PARTIAL_REVERSAL: Stim-OFF vs Stim-ON ──────────────────
# Scope: wrong trials only (DIFFERENT_NUMBER excluded)
# y-axis: count of PARTIAL_REVERSAL trials in each stim group
# Layout: 1×3  (S2 | S3 | Combined)  WITH stats boxes

def plot_error_stim_comparison(error_type, title_label, colour_on, colour_off,
                                fname, y_label='Number of Trials'):
    fig, axes = plt.subplots(1, 3, figsize=(13, 5.5), facecolor='white')
    fig.subplots_adjust(wspace=0.42, top=0.78, bottom=0.13, left=0.07, right=0.97)

    for ax, (sess_label, trials, n_total) in zip(axes, sessions_info):
        wrong = [t for t in trials if t['error_type'] in ('PARTIAL_REVERSAL', 'OTHER')]
        no_stim = [t for t in wrong if not t['stim_present_condB']]
        stim_on = [t for t in wrong if     t['stim_present_condB']]

        n_no = sum(1 for t in no_stim if t['error_type'] == error_type)
        n_on = sum(1 for t in stim_on if t['error_type'] == error_type)

        # Vectors for Fisher's test (1 = this error type, 0 = other wrong)
        vec_no = [1 if t['error_type'] == error_type else 0 for t in no_stim]
        vec_on = [1 if t['error_type'] == error_type else 0 for t in stim_on]
        res = run_fisher_comparison(vec_no, vec_on, sess_label)

        vals   = [n_no, n_on]
        colors = [colour_off, colour_on]
        total_no = len(no_stim)
        total_on = len(stim_on)
        totals   = [total_no, total_on]

        ax.bar([0, 1], vals, width=0.5, color=colors,
               edgecolor='white', linewidth=1.4, zorder=3)

        for xi, (v, tot, col) in enumerate(zip(vals, totals, colors)):
            pct = (100 * v / tot) if tot > 0 else 0
            lbl = f'{v}/{tot}'
            if v >= 2:
                ax.text(xi, v / 2, lbl, ha='center', va='center',
                        fontsize=9, fontweight='bold', color='white', zorder=8)
            elif v > 0:
                ax.text(xi, v + 0.2, lbl, ha='center', va='bottom',
                        fontsize=8.5, fontweight='bold', color=col)
            else:
                ax.text(xi, 0.2, '0', ha='center', va='bottom',
                        fontsize=9, color='#aaa')

        ax.set_xticks([0, 1])
        ax.set_xticklabels(['NO-STIM', 'STIM-ON'], fontsize=10, fontweight='bold')
        max_y = max(max(vals) + 2, 5)
        ax.set_ylim(0, max_y)
        ax.set_yticks(range(0, max_y + 1, 1))
        ax.set_ylabel(y_label, fontsize=9)
        ax.yaxis.grid(True, color='#eeeeee', zorder=0)
        ax.set_axisbelow(True)
        ax.set_facecolor('#FAFAFA')
        ax.set_title(sess_label, fontsize=11, fontweight='bold', color='#1A1A2E', pad=8)

        # Stats box
        # _stat_box(ax, res, 1.13, fontsize=8.5)

    fig.suptitle(f'{title_label}: NO-STIM vs STIM-ON',
                 fontsize=13, fontweight='bold', color='#111', y=0.97)
    fig.text(0.5, 0.87,
             f'Condition B  |  mA ≥ {STIM_THRESHOLD}  |  '
             'Scope: wrong trials only (DIFFERENT_NUMBER excluded)  |  '
             'Fisher p + Cohen d + effect',
             ha='center', fontsize=8.5, color='#555')

    fig.legend(handles=[
        mpatches.Patch(facecolor=colour_off, label='NO-STIM'),
        mpatches.Patch(facecolor=colour_on,  label='STIM-ON'),
    ], loc='lower center', ncol=2, fontsize=10, framealpha=0.97,
       facecolor='white', edgecolor='#ccc', bbox_to_anchor=(0.5, 0.01))

    save_fig(fig, COMBINED_DIR / fname)


# PARTIAL_REVERSAL
plot_error_stim_comparison(
    error_type   = 'PARTIAL_REVERSAL',
    title_label  = 'Plot 3 — Partial Reversal Errors',
    colour_on    = C_STIM,
    colour_off   = C_NO_STIM,
    fname        = 'plot3_partial_reversal_stim_condB.png',
    y_label      = 'Partial Reversal Count',
)

In [ ]:
# ── Cell 11: Plot 4 — OTHER Errors: Stim-OFF vs Stim-ON ──────────────────────

plot_error_stim_comparison(
    error_type   = 'OTHER',
    title_label  = 'Plot 4 — Other Errors (no reversal pattern)',
    colour_on    = '#7B1FA2',   # darker purple for stim-on
    colour_off   = '#CE93D8',   # lighter purple for no-stim
    fname        = 'plot4_other_errors_stim_condB.png',
    y_label      = 'Other Error Count',
)

In [ ]:
# ── Cell 12: Plot 5 — Error Composition by Stim Group (Stacked Bar) ──────────
# For each session × stim group: stacked bar showing PARTIAL_REVERSAL + OTHER
# as proportion of total wrong trials in that group.
# Layout: 1×3  (S2 | S3 | Combined)
# No stats on this plot — it's a composition view

fig, axes = plt.subplots(1, 3, figsize=(13, 5.5), facecolor='white')
fig.subplots_adjust(wspace=0.42, top=0.82, bottom=0.13, left=0.07, right=0.97)

for ax, (sess_label, trials, n_total) in zip(axes, sessions_info):
    wrong = [t for t in trials if t['error_type'] in ('PARTIAL_REVERSAL', 'OTHER')]
    no_stim = [t for t in wrong if not t['stim_present_condB']]
    stim_on = [t for t in wrong if     t['stim_present_condB']]

    groups   = [no_stim, stim_on]
    xlabels  = ['NO-STIM', 'STIM-ON']
    x_pos    = [0, 1]

    for xi, (grp, xl) in enumerate(zip(groups, xlabels)):
        n_grp     = len(grp)
        n_partial = sum(1 for t in grp if t['error_type'] == 'PARTIAL_REVERSAL')
        n_other   = sum(1 for t in grp if t['error_type'] == 'OTHER')

        if n_grp == 0:
            ax.bar(xi, 0, width=0.5, color='#eee', edgecolor='white')
            ax.text(xi, 0.3, 'n=0', ha='center', va='bottom', fontsize=8, color='#aaa')
            continue

        # Stacked bar: PARTIAL on bottom, OTHER on top
        ax.bar(xi, n_partial, width=0.5, color=C_PARTIAL,
               edgecolor='white', linewidth=1.4, zorder=3, label='Partial Reversal')
        ax.bar(xi, n_other,   width=0.5, color=C_OTHER,
               edgecolor='white', linewidth=1.4, zorder=3, bottom=n_partial,
               label='Other Error')

        # Label partial segment
        if n_partial > 0:
            pct_p = 100 * n_partial / n_grp
            ax.text(xi, n_partial / 2,
                    f'{n_partial}',
                    ha='center', va='center', fontsize=8.5,
                    fontweight='bold', color='white', zorder=8)

        # Label other segment
        if n_other > 0:
            pct_o = 100 * n_other / n_grp
            ax.text(xi, n_partial + n_other / 2,
                    f'{n_other}',
                    ha='center', va='center', fontsize=8.5,
                    fontweight='bold', color='white', zorder=8)

        # Total label above bar
        ax.text(xi, n_grp + 0.2, f'n={n_grp}',
                ha='center', va='bottom', fontsize=9,
                fontweight='bold', color='#333')

    ax.set_xticks(x_pos)
    ax.set_xticklabels(xlabels, fontsize=7, fontweight='bold')
    max_wrong = max(
        len([t for t in wrong if not t['stim_present_condB']]),
        len([t for t in wrong if     t['stim_present_condB']]),
        1
    )
    ax.set_ylim(0, max_wrong + 3)
    ax.set_yticks(range(0, max_wrong + 3))
    ax.set_ylabel('Wrong Trial Count', fontsize=9)
    ax.yaxis.grid(True, color='#eeeeee', zorder=0)
    ax.set_axisbelow(True)
    ax.set_facecolor('#FAFAFA')
    ax.set_title(sess_label, fontsize=11, fontweight='bold', color='#1A1A2E', pad=8)

fig.suptitle('Plot 5 — Error Composition by Stim Group (Stacked)',
             fontsize=13, fontweight='bold', color='#111', y=0.97)
fig.text(0.5, 0.88,
         'Condition B  |  mA ≥ 2.0  |  '
         'Wrong trials only (DIFFERENT_NUMBER excluded)  |  No statistics',
         ha='center', fontsize=8.5, color='#555')

fig.legend(handles=[
    mpatches.Patch(facecolor=C_PARTIAL, label='Partial Reversal'),
    mpatches.Patch(facecolor=C_OTHER,   label='Other Error'),
], loc='lower center', ncol=2, fontsize=10, framealpha=0.97,
   facecolor='white', edgecolor='#ccc', bbox_to_anchor=(0.5, 0.01))

save_fig(fig, COMBINED_DIR / 'plot5_error_composition_stacked_condB.png')

In [ ]:
# ── Cell 13: Plot 6 — All 4 Combinations: PARTIAL/OTHER × NO-STIM/STIM-ON ────
# The 4 groups side by side per session panel:
#   (1) PARTIAL  NO-STIM   (2) PARTIAL  STIM-ON
#   (3) OTHER    NO-STIM   (4) OTHER    STIM-ON
#
# Layout: 1×3  (S2 | S3 | Combined)
# Stats: Fisher + Cohen's d for PARTIAL(NO vs ON) and OTHER(NO vs ON)
#        shown as two separate stat boxes per panel

fig, axes = plt.subplots(1, 3, figsize=(15, 6), facecolor='white')
fig.subplots_adjust(wspace=0.42, top=0.76, bottom=0.14, left=0.07, right=0.97)

# Colours for the 4 groups
C_P_NO = '#FFAB76'   # light orange — partial no-stim
C_P_ON = '#E65100'   # dark orange  — partial stim-on
C_O_NO = '#CE93D8'   # light purple — other no-stim
C_O_ON = '#6A1B9A'   # dark purple  — other stim-on

X_POS    = [0, 1, 2.6, 3.6]   # gap between the two pairs
XLABELS  = ['PARTIAL\nNO-STIM', 'PARTIAL\nSTIM-ON', 'OTHER\nNO-STIM', 'OTHER\nSTIM-ON']
COLORS   = [C_P_NO, C_P_ON, C_O_NO, C_O_ON]
PAIR_ERR = ['PARTIAL_REVERSAL', 'PARTIAL_REVERSAL', 'OTHER', 'OTHER']
PAIR_STM = [False, True, False, True]   # False = NO-STIM, True = STIM-ON

for ax, (sess_label, trials, n_total) in zip(axes, sessions_info):
    wrong = [t for t in trials if t['error_type'] in ('PARTIAL_REVERSAL', 'OTHER')]
    no_stim = [t for t in wrong if not t['stim_present_condB']]
    stim_on = [t for t in wrong if     t['stim_present_condB']]

    # Pool for lookup: (error_type, stim_bool) -> list of trials
    pool = {
        ('PARTIAL_REVERSAL', False): [t for t in no_stim if t['error_type'] == 'PARTIAL_REVERSAL'],
        ('PARTIAL_REVERSAL', True ): [t for t in stim_on if t['error_type'] == 'PARTIAL_REVERSAL'],
        ('OTHER',            False): [t for t in no_stim if t['error_type'] == 'OTHER'],
        ('OTHER',            True ): [t for t in stim_on if t['error_type'] == 'OTHER'],
    }

    # Denominator for each bar: total trials in that stim group (all wrong included)
    denom = {False: len(no_stim), True: len(stim_on)}

    vals = [len(pool[(et, s)]) for et, s in zip(PAIR_ERR, PAIR_STM)]

    bars = ax.bar(X_POS, vals, width=0.7, color=COLORS,
                  edgecolor='white', linewidth=1.4, zorder=3)

    for xi, (v, et, stim_b, col) in enumerate(zip(vals, PAIR_ERR, PAIR_STM, COLORS)):
        tot = denom[stim_b]
        pct = (100 * v / tot) if tot > 0 else 0
        lbl = f'{v}/{tot}'
        if v >= 2:
            ax.text(X_POS[xi], v / 2, lbl, ha='center', va='center',
                    fontsize=8.5, fontweight='bold', color='black', zorder=8)
        elif v > 0:
            ax.text(X_POS[xi], v + 0.15, lbl, ha='center', va='bottom',
                    fontsize=8, fontweight='bold', color=col)
        else:
            ax.text(X_POS[xi], 0.15, '0', ha='center', va='bottom',
                    fontsize=9, color='#aaa')

    # ── Stats: PARTIAL NO-STIM vs STIM-ON ─────────────────────────────────
    vec_p_no = [1 if t['error_type'] == 'PARTIAL_REVERSAL' else 0 for t in no_stim]
    vec_p_on = [1 if t['error_type'] == 'PARTIAL_REVERSAL' else 0 for t in stim_on]
    res_p = run_fisher_comparison(vec_p_no, vec_p_on, sess_label)

    # ── Stats: OTHER NO-STIM vs STIM-ON ───────────────────────────────────
    vec_o_no = [1 if t['error_type'] == 'OTHER' else 0 for t in no_stim]
    vec_o_on = [1 if t['error_type'] == 'OTHER' else 0 for t in stim_on]
    res_o = run_fisher_comparison(vec_o_no, vec_o_on, sess_label)

    def _fmt_statbox(res):
        p_v, d_v = res['p_value'], res['cohens_d']
        notable = ((not np.isnan(p_v) and p_v < 0.05) or
                   (not np.isnan(d_v) and abs(d_v) >= 0.8))
        fc = '#FFF0F0' if notable else '#F5F5F5'
        ec = '#C62828' if notable else '#888888'
        p_s = f'p={p_v:.3f}' if not np.isnan(p_v) else 'p=N/A'
        d_s = f'd={d_v:+.2f}' if not np.isnan(d_v) else 'd=N/A'
        sig = sig_stars(p_v)
        txt = f'{sig}  {p_s}  {d_s}  [{effect_label(d_v)}]'
        return txt, fc, ec

    # Stat box for PARTIAL pair (centred over x=0.5)
    # txt_p, fc_p, ec_p = _fmt_statbox(res_p)
    # ax.text(0.5, 1.12, f'Partial: {txt_p}',
    #         transform=ax.transAxes, ha='center', va='center',
    #         fontsize=7.8, fontweight='bold', color=ec_p,
    #         bbox=dict(boxstyle='round,pad=0.3', fc=fc_p, ec=ec_p, lw=1.1, alpha=0.97))

    # # Stat box for OTHER pair (centred over x=3.1)
    # txt_o, fc_o, ec_o = _fmt_statbox(res_o)
    # ax.text(0.5, 1.22, f'Other:   {txt_o}',
    #         transform=ax.transAxes, ha='center', va='center',
    #         fontsize=7.8, fontweight='bold', color=ec_o,
    #         bbox=dict(boxstyle='round,pad=0.3', fc=fc_o, ec=ec_o, lw=1.1, alpha=0.97))

    # Bracket lines separating the two pairs
    max_y = max(max(vals) + 1, 4)
    ax.axvline(x=1.8, color='#cccccc', linewidth=1.2, linestyle='--', zorder=1)

    ax.set_xticks(X_POS)
    ax.set_xticklabels(XLABELS, fontsize=8.5, fontweight='bold')
    ax.set_ylim(0, max_y + 2)
    ax.set_yticks(range(0, max_y + 2))
    ax.set_ylabel('Count (within stim group)', fontsize=9)
    ax.yaxis.grid(True, color='#eeeeee', zorder=0)
    ax.set_axisbelow(True)
    ax.set_facecolor('#FAFAFA')
    ax.set_title(sess_label, fontsize=11, fontweight='bold', color='#1A1A2E', pad=8)

    # Group labels above pairs
    ax.text(0.5,  max_y + 1.5, 'Partial Reversal', ha='center', va='center',
            fontsize=8.5, color=C_P_ON, fontweight='bold')
    ax.text(3.1, max_y + 1.5, 'Other Error',      ha='center', va='center',
            fontsize=8.5, color=C_O_ON, fontweight='bold')

fig.suptitle('Plot 6 — All 4 Combinations: Partial Reversal & Other × Stim-OFF & Stim-ON',
             fontsize=12, fontweight='bold', color='#111', y=0.99)
fig.text(0.5, 0.86,
         'Condition B  |  mA ≥ 2.0  |  Scope: wrong trials only (DIFFERENT_NUMBER excluded)  |  '
         'Denom = total wrong trials in that stim group  |  No Stats',
         ha='center', fontsize=7, color='#444')

fig.legend(handles=[
    mpatches.Patch(facecolor=C_P_NO, label='Partial  NO-STIM'),
    mpatches.Patch(facecolor=C_P_ON, label='Partial  STIM-ON'),
    mpatches.Patch(facecolor=C_O_NO, label='Other    NO-STIM'),
    mpatches.Patch(facecolor=C_O_ON, label='Other    STIM-ON'),
], loc='lower center', ncol=4, fontsize=9, framealpha=0.97,
   facecolor='white', edgecolor='#ccc', bbox_to_anchor=(0.5, 0.01))

save_fig(fig, COMBINED_DIR / 'plot6_all4_combinations_condB.png')


In [ ]:
# ── Cell 14: Plot 7 — Error Types × Stim Group × Difficulty (Easy=2+3d, Hard=4+5d) ──
# 8 combinations per session:
#   Easy: PARTIAL NO-STIM | PARTIAL STIM-ON | OTHER NO-STIM | OTHER STIM-ON
#   Hard: PARTIAL NO-STIM | PARTIAL STIM-ON | OTHER NO-STIM | OTHER STIM-ON
#
# Layout: one figure per session scope (S2, S3, Combined)
#         each figure = 1×2 panels (Easy | Hard)
# Stats:  two stat boxes per panel (Partial pair + Other pair)
# Condition B  |  mA >= 2.0  |  Wrong trials only (DIFFERENT_NUMBER excluded)

EASY_SET = {2, 3}
HARD_SET  = {4, 5}

# ── Colours (consistent with Plot 6) ─────────────────────────────────────────
C_P_NO = '#FFAB76'   # light orange — partial no-stim
C_P_ON = '#E65100'   # dark orange  — partial stim-on
C_O_NO = '#CE93D8'   # light purple — other no-stim
C_O_ON = '#6A1B9A'   # dark purple  — other stim-on

BAR_COLORS  = [C_P_NO, C_P_ON, C_O_NO, C_O_ON]
BAR_LABELS  = ['Partial\nNO-STIM', 'Partial\nSTIM-ON', 'Other\nNO-STIM', 'Other\nSTIM-ON']
PAIR_ERRS   = ['PARTIAL_REVERSAL', 'PARTIAL_REVERSAL', 'OTHER', 'OTHER']
PAIR_STIMS  = [False, True, False, True]

# x positions: gap of 1.4 between the two error-type pairs
X_POS = [0, 0.8, 2.2, 3.0]


def _fmt_statbox(res):
    p_v, d_v = res['p_value'], res['cohens_d']
    notable  = ((not np.isnan(p_v) and p_v < 0.05) or
                (not np.isnan(d_v) and abs(d_v) >= 0.8))
    fc  = '#FFF0F0' if notable else '#F5F5F5'
    ec  = '#C62828' if notable else '#888888'
    p_s = f'p={p_v:.3f}' if not np.isnan(p_v) else 'p=N/A'
    d_s = f'd={d_v:+.2f}' if not np.isnan(d_v) else 'd=N/A'
    sig = sig_stars(p_v)
    eff = effect_label(d_v)
    return f'{sig} {p_s}\n{d_s} [{eff}]', fc, ec


def plot_diff_panel(ax, wrong_trials, diff_label, diff_color):
    """
    Draw one difficulty panel (Easy or Hard) on ax.
    wrong_trials : list of trial dicts already filtered to the right digit-set
                   and already excluding DIFFERENT_NUMBER.
    """
    no_stim = [t for t in wrong_trials if not t['stim_present_condB']]
    stim_on = [t for t in wrong_trials if     t['stim_present_condB']]
    denom   = {False: len(no_stim), True: len(stim_on)}

    pool = {
        ('PARTIAL_REVERSAL', False): [t for t in no_stim if t['error_type'] == 'PARTIAL_REVERSAL'],
        ('PARTIAL_REVERSAL', True ): [t for t in stim_on if t['error_type'] == 'PARTIAL_REVERSAL'],
        ('OTHER',            False): [t for t in no_stim if t['error_type'] == 'OTHER'],
        ('OTHER',            True ): [t for t in stim_on if t['error_type'] == 'OTHER'],
    }

    vals = [len(pool[(et, s)]) for et, s in zip(PAIR_ERRS, PAIR_STIMS)]
    max_y = max(max(vals) + 1, 4)

    bars = ax.bar(X_POS, vals, width=0.65, color=BAR_COLORS,
                  edgecolor='white', linewidth=1.3, zorder=3)

    # ── Bar labels ────────────────────────────────────────────────────────────
    for xi, (v, et, stim_b, col) in enumerate(zip(vals, PAIR_ERRS, PAIR_STIMS, BAR_COLORS)):
        tot = denom[stim_b]
        pct = (100 * v / tot) if tot > 0 else 0
        lbl = f'{v}/{tot}'
        if v >= 2:
            ax.text(X_POS[xi], v / 2, lbl,
                    ha='center', va='center',
                    fontsize=7.5, fontweight='bold', color='black',
                    linespacing=1.3, zorder=8)
        elif v == 1:
            ax.text(X_POS[xi], v + 0.12, lbl,
                    ha='center', va='bottom',
                    fontsize=7, fontweight='bold', color=col,
                    linespacing=1.3)
        else:
            ax.text(X_POS[xi], 0.12, '0',
                    ha='center', va='bottom',
                    fontsize=8, color='#aaa')

    # ── Stat boxes (stacked, above panel) ────────────────────────────────────
    vec_p_no = [1 if t['error_type'] == 'PARTIAL_REVERSAL' else 0 for t in no_stim]
    vec_p_on = [1 if t['error_type'] == 'PARTIAL_REVERSAL' else 0 for t in stim_on]
    vec_o_no = [1 if t['error_type'] == 'OTHER'            else 0 for t in no_stim]
    vec_o_on = [1 if t['error_type'] == 'OTHER'            else 0 for t in stim_on]

    res_p = run_fisher_comparison(vec_p_no, vec_p_on, '')
    res_o = run_fisher_comparison(vec_o_no, vec_o_on, '')

    # txt_p, fc_p, ec_p = _fmt_statbox(res_p)
    # txt_o, fc_o, ec_o = _fmt_statbox(res_o)

    # # Partial stat box — left side, above bars 0-1
    # ax.text(0.4, 1.08, f'Partial:\n{txt_p}',
    #         transform=ax.transAxes, ha='center', va='bottom',
    #         fontsize=6.8, fontweight='bold', color=ec_p,
    #         linespacing=1.35,
    #         bbox=dict(boxstyle='round,pad=0.28', fc=fc_p, ec=ec_p, lw=1.0, alpha=0.97))

    # # Other stat box — right side, above bars 2-3
    # ax.text(0.85, 1.08, f'Other:\n{txt_o}',
    #         transform=ax.transAxes, ha='center', va='bottom',
    #         fontsize=6.8, fontweight='bold', color=ec_o,
    #         linespacing=1.35,
    #         bbox=dict(boxstyle='round,pad=0.28', fc=fc_o, ec=ec_o, lw=1.0, alpha=0.97))

    # ── Divider between the two error-type pairs ──────────────────────────────
    ax.axvline(x=1.5, color='#cccccc', linewidth=1.0, linestyle='--', zorder=1)

    # ── Group header labels ───────────────────────────────────────────────────
    ax.text(0.4,  max_y + 0.65, 'Partial Reversal',
            ha='center', va='center', fontsize=7.5,
            color=C_P_ON, fontweight='bold')
    ax.text(2.6, max_y + 0.65, 'Other Error',
            ha='center', va='center', fontsize=7.5,
            color=C_O_ON, fontweight='bold')

    # ── n annotations (group sizes) ───────────────────────────────────────────
    ax.text(3.8, 0.3,
            f'NO-STIM n={len(no_stim)}\nSTIM-ON n={len(stim_on)}',
            ha='right', va='bottom', fontsize=6.5, color='#555',
            linespacing=1.4,
            bbox=dict(boxstyle='round,pad=0.25', fc='#F5F5F5', ec='#ccc', lw=0.8))

    ax.set_xticks(X_POS)
    ax.set_xticklabels(BAR_LABELS,
                       fontsize=7.5, fontweight='bold',
                       linespacing=1.3)
    ax.set_ylim(0, max_y + 1.5)
    ax.set_yticks(range(0, max_y + 2))
    ax.set_ylabel('Count (within stim group)', fontsize=8)
    ax.yaxis.grid(True, color='#eeeeee', zorder=0)
    ax.set_axisbelow(True)
    ax.set_facecolor('#FAFAFA')

    easy_str = '+'.join(str(d) for d in sorted(EASY_SET))
    hard_str = '+'.join(str(d) for d in sorted(HARD_SET))
    diff_str = easy_str if diff_label == 'Easy' else hard_str
    ax.set_title(f'{diff_label}  ({diff_str} digits)  —  n={len(wrong_trials)} wrong',
                 fontsize=9.5, fontweight='bold', color=diff_color, pad=6)


def plot_difficulty_split(trials, scope_label, fname):
    """
    One figure with 2 panels: Easy | Hard
    trials : full trial list for the scope (already has error_type annotated)
    """
    wrong = [t for t in trials if t['error_type'] in ('PARTIAL_REVERSAL', 'OTHER')]

    easy_wrong = [t for t in wrong if t['digits'] in EASY_SET]
    hard_wrong = [t for t in wrong if t['digits'] in HARD_SET]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), facecolor='white')
    fig.subplots_adjust(wspace=0.45, top=0.72, bottom=0.16,
                        left=0.07, right=0.97)

    plot_diff_panel(axes[0], easy_wrong, 'Easy', '#1565C0')
    plot_diff_panel(axes[1], hard_wrong, 'Hard', '#BF360C')

    fig.suptitle(
        f'Plot 7 — Error Types × Stim Group × Difficulty  |  {scope_label}',
        fontsize=12, fontweight='bold', color='#111', y=0.99)
    fig.text(
        0.5, 0.89,
        'Condition B  |  mA ≥ 2.0  |  Wrong trials only (DIFFERENT_NUMBER excl.)  |  '
        'Easy = 2+3 digits  |  Hard = 4+5 digits  |  No Stats',
        ha='center', fontsize=7.8, color='#555')

    fig.legend(handles=[
        mpatches.Patch(facecolor=C_P_NO, label='Partial  NO-STIM'),
        mpatches.Patch(facecolor=C_P_ON, label='Partial  STIM-ON'),
        mpatches.Patch(facecolor=C_O_NO, label='Other    NO-STIM'),
        mpatches.Patch(facecolor=C_O_ON, label='Other    STIM-ON'),
    ], loc='lower center', ncol=4, fontsize=8.5, framealpha=0.97,
       facecolor='white', edgecolor='#ccc', bbox_to_anchor=(0.5, 0.01))

    save_fig(fig, COMBINED_DIR / fname)


# ── Run for all three scopes ──────────────────────────────────────────────────
plot_difficulty_split(all_s2,       'Session 2',         'plot7_diff_split_s2_condB.png')
plot_difficulty_split(all_s3,       'Session 3',         'plot7_diff_split_s3_condB.png')
plot_difficulty_split(all_combined, 'Combined (S2+S3)',  'plot7_diff_split_combined_condB.png')

In [ ]:
# ── Cell 14: Plot 7 — Error Types × Stim Group × Difficulty (Easy=2+3d, Hard=4+5d) ──
# Layout: ONE figure — 2 rows × 3 columns
#   Row 1: Easy  |  Col 1: S2  |  Col 2: S3  |  Col 3: S2+S3
#   Row 2: Hard  |  Col 1: S2  |  Col 2: S3  |  Col 3: S2+S3
# Condition B  |  mA >= 2.0  |  Wrong trials only (DIFFERENT_NUMBER excluded)

EASY_SET = {2, 3}
HARD_SET  = {4, 5}

# ── Colours ───────────────────────────────────────────────────────────────────
C_P_NO = '#FFAB76'
C_P_ON = '#E65100'
C_O_NO = '#CE93D8'
C_O_ON = '#6A1B9A'

BAR_COLORS  = [C_P_NO, C_P_ON, C_O_NO, C_O_ON]
BAR_LABELS  = ['Partial\nNO-STIM', 'Partial\nSTIM-ON', 'Other\nNO-STIM', 'Other\nSTIM-ON']
PAIR_ERRS   = ['PARTIAL_REVERSAL', 'PARTIAL_REVERSAL', 'OTHER', 'OTHER']
PAIR_STIMS  = [False, True, False, True]

X_POS = [0, 0.8, 2.2, 3.0]


def _fmt_statbox(res):
    p_v, d_v = res['p_value'], res['cohens_d']
    notable  = ((not np.isnan(p_v) and p_v < 0.05) or
                (not np.isnan(d_v) and abs(d_v) >= 0.8))
    fc  = '#FFF0F0' if notable else '#F5F5F5'
    ec  = '#C62828' if notable else '#888888'
    p_s = f'p={p_v:.3f}' if not np.isnan(p_v) else 'p=N/A'
    d_s = f'd={d_v:+.2f}' if not np.isnan(d_v) else 'd=N/A'
    sig = sig_stars(p_v)
    eff = effect_label(d_v)
    return f'{sig} {p_s}\n{d_s} [{eff}]', fc, ec


def plot_diff_panel(ax, wrong_trials, diff_label, diff_color, show_ylabel=True):
    """
    Draw one panel on ax.
    wrong_trials : list of trial dicts already filtered to the right digit-set
                   and already excluding DIFFERENT_NUMBER.
    show_ylabel  : only draw y-axis label on leftmost column.
    """
    no_stim = [t for t in wrong_trials if not t['stim_present_condB']]
    stim_on = [t for t in wrong_trials if     t['stim_present_condB']]
    denom   = {False: len(no_stim), True: len(stim_on)}

    pool = {
        ('PARTIAL_REVERSAL', False): [t for t in no_stim if t['error_type'] == 'PARTIAL_REVERSAL'],
        ('PARTIAL_REVERSAL', True ): [t for t in stim_on if t['error_type'] == 'PARTIAL_REVERSAL'],
        ('OTHER',            False): [t for t in no_stim if t['error_type'] == 'OTHER'],
        ('OTHER',            True ): [t for t in stim_on if t['error_type'] == 'OTHER'],
    }

    vals  = [len(pool[(et, s)]) for et, s in zip(PAIR_ERRS, PAIR_STIMS)]
    max_y = max(max(vals) + 1, 4)

    ax.bar(X_POS, vals, width=0.65, color=BAR_COLORS,
           edgecolor='white', linewidth=1.3, zorder=3)

    # ── Bar labels ────────────────────────────────────────────────────────────
    for xi, (v, et, stim_b, col) in enumerate(zip(vals, PAIR_ERRS, PAIR_STIMS, BAR_COLORS)):
        tot = denom[stim_b]
        lbl = f'{v}/{tot}'
        if v >= 2:
            ax.text(X_POS[xi], v / 2, lbl,
                    ha='center', va='center',
                    fontsize=7.5, fontweight='bold', color='black',
                    linespacing=1.3, zorder=8)
        elif v == 1:
            ax.text(X_POS[xi], v + 0.12, lbl,
                    ha='center', va='bottom',
                    fontsize=7, fontweight='bold', color=col,
                    linespacing=1.3)
        else:
            ax.text(X_POS[xi], 0.12, '0',
                    ha='center', va='bottom',
                    fontsize=8, color='#aaa')

    # ── Divider between the two error-type pairs ──────────────────────────────
    ax.axvline(x=1.5, color='#cccccc', linewidth=1.0, linestyle='--', zorder=1)

    # ── Group header labels ───────────────────────────────────────────────────
    ax.text(0.4,  max_y + 0.45, 'Partial Reversal',
            ha='center', va='center', fontsize=7,
            color=C_P_ON, fontweight='bold')
    ax.text(2.6, max_y + 0.45, 'Other Error',
            ha='center', va='center', fontsize=7,
            color=C_O_ON, fontweight='bold')

    # ── n annotations (group sizes) ───────────────────────────────────────────
    ax.text(3.8, 0.3,
            f'NO-STIM n={len(no_stim)}\nSTIM-ON n={len(stim_on)}',
            ha='right', va='bottom', fontsize=6.2, color='#555',
            linespacing=1.4,
            bbox=dict(boxstyle='round,pad=0.25', fc='#F5F5F5', ec='#ccc', lw=0.8))

    ax.set_xticks(X_POS)
    ax.set_xticklabels(BAR_LABELS, fontsize=7, fontweight='bold', linespacing=1.3)
    ax.set_ylim(0, max_y + 1.2)
    ax.set_yticks(range(0, max_y + 2))
    if show_ylabel:
        ax.set_ylabel('Count (within stim group)', fontsize=8)
    ax.yaxis.grid(True, color='#eeeeee', zorder=0)
    ax.set_axisbelow(True)
    ax.set_facecolor('#FAFAFA')

    easy_str = '+'.join(str(d) for d in sorted(EASY_SET))
    hard_str = '+'.join(str(d) for d in sorted(HARD_SET))
    diff_str = easy_str if diff_label == 'Easy' else hard_str
    ax.set_title(f'{diff_label}  ({diff_str} digits)  —  n={len(wrong_trials)} wrong',
                 fontsize=8.5, fontweight='bold', color=diff_color, pad=5)


def plot_difficulty_split_grid(trials_s2, trials_s3, trials_combined, fname):
    """
    Single figure: 2 rows × 3 columns
      Row 0 = Easy,  Row 1 = Hard
      Col 0 = S2,    Col 1 = S3,    Col 2 = Combined
    """
    scope_data   = [trials_s2, trials_s3, trials_combined]
    scope_labels = ['Session 2', 'Session 3', 'Combined (S2+S3)']
    diff_sets    = [EASY_SET, HARD_SET]
    diff_labels  = ['Easy', 'Hard']
    diff_colors  = ['#1565C0', '#BF360C']

    fig, axes = plt.subplots(2, 3, figsize=(18, 9), facecolor='white')
    fig.subplots_adjust(wspace=0.38, hspace=0.52,
                        top=0.88, bottom=0.12,
                        left=0.06, right=0.98)

    for row, (diff_label, diff_set, diff_color) in enumerate(
            zip(diff_labels, diff_sets, diff_colors)):
        for col, (trials, scope_label) in enumerate(
                zip(scope_data, scope_labels)):

            ax = axes[row, col]

            # Filter: wrong trials only, correct digit-set
            wrong = [t for t in trials
                     if t['error_type'] in ('PARTIAL_REVERSAL', 'OTHER')
                     and t['digits'] in diff_set]

            show_ylabel = (col == 0)
            plot_diff_panel(ax, wrong, diff_label, diff_color,
                            show_ylabel=show_ylabel)

            # Column header (only on top row)
            if row == 0:
                ax.text(0.5, 1.18, scope_label,
                        transform=ax.transAxes,
                        ha='center', va='bottom',
                        fontsize=10, fontweight='bold', color='#222')

    # ── Row labels on the far left ────────────────────────────────────────────
    for row, (diff_label, diff_color) in enumerate(zip(diff_labels, diff_colors)):
        fig.text(0.005, axes[row, 0].get_position().y0 +
                 axes[row, 0].get_position().height / 2,
                 diff_label, ha='left', va='center',
                 fontsize=11, fontweight='bold', color=diff_color,
                 rotation=90)

    fig.suptitle(
        'Plot 7 — Error Types × Stim Group × Difficulty',
        fontsize=13, fontweight='bold', color='#111', y=0.97)
    # fig.text(
    #     0.5, 0.925,
    #     'Condition B  |  mA ≥ 2.0  |  Wrong trials only (DIFFERENT_NUMBER excl.)  |  '
    #     'Easy = 2+3 digits  |  Hard = 4+5 digits  |  No Stats',
    #     ha='center', fontsize=8, color='#555')

    fig.legend(handles=[
        mpatches.Patch(facecolor=C_P_NO, label='Partial  NO-STIM'),
        mpatches.Patch(facecolor=C_P_ON, label='Partial  STIM-ON'),
        mpatches.Patch(facecolor=C_O_NO, label='Other    NO-STIM'),
        mpatches.Patch(facecolor=C_O_ON, label='Other    STIM-ON'),
    ], loc='lower center', ncol=4, fontsize=9, framealpha=0.97,
       facecolor='white', edgecolor='#ccc', bbox_to_anchor=(0.5, 0.005))

    save_fig(fig, COMBINED_DIR / fname)


# ── Run once ──────────────────────────────────────────────────────────────────
plot_difficulty_split_grid(
    all_s2,
    all_s3,
    all_combined,
    'plot7_diff_split_grid_condB.png'
)